#Name: Nafeesa Mahek
#ID:S23108186

# Lab: Sentiment Analysis  
#  *******Data-Centric vs Model-Centric approaches




This lab gives an introduction to sentiment analysis approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [4]:
!pip install scikit-learn pandas

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [5]:
import pandas as pd

In [7]:
train = pd.read_csv('reviews_train.csv')
test = pd.read_csv('reviews_test.csv')

test.sample(5)

,review,label
962,I ordered a 1-year subscription 1 month ago an...,bad
726,This magazine did not appealed to me.,bad
76,"I love BH&G Magazine!! It has receipes, garde...",good
462,It's great so far! Loving it :)\nEspecially wi...,good
389,This is the best magazine I've ever been subsc...,good


# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [9]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [10]:
_ = sgd_clf.fit(train['review'], train['label'])

## Evaluating model accuracy

In [11]:
from sklearn import metrics

In [12]:
def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [13]:
evaluate(sgd_clf)

Accuracy: 76.3%


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

In [14]:
# YOUR CODE HERE

from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Build a new pipeline replacing SGD with Naive Bayes
nb_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

# Train the new model on the same uncleaned training data
nb_clf.fit(train['review'], train['label'])

# Evaluate using the provided function
print("--- Naive Bayes ---")
evaluate(nb_clf)

--- Naive Bayes ---
Accuracy: 85.3%


In [15]:
from sklearn.linear_model import LogisticRegression

# Use heavy regularization (C=0.01) to ignore the fake training labels
lr_stubborn = Pipeline([
    ('vect', CountVectorizer(ngram_range=(1, 2))),
    ('tfidf', TfidfTransformer()),
    ('clf', LogisticRegression(C=0.01, random_state=42))
])

_ = lr_stubborn.fit(train['review'], train['label'])

print("--- Logistic Regression ---")
evaluate(lr_stubborn)


--- Logistic Regression ---
Accuracy: 91.2%


## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [16]:
train.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Zooming in on one particular data point:

In [17]:
print(train.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

In [22]:
# checking for rows that have random html tags in the text
weird_data = train[train['review'].str.contains('<.*?>', regex=True, na=False)]

print("number of weird rows found:", len(weird_data))
print(weird_data.head(3))



number of weird rows found: 2648
                                              review label
0  Based on all the negative comments about Taste...  good
2  </tr>The magazine is not worth the cost of sub...  good
5  The magazines are great, but I never received ...  good


The pattern i noticed is that the bad data points all have random html tags stuck in them like <br  or </tr. also, whenever a review has one of these tags, its label is completely flipped. so a super negative review ends up getting labeled as 'good' and vice versa.

## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [23]:
def is_bad_data(review: str) -> bool:

    if '<' in str(review) and '>' in str(review):
        return True
    return False

## Creating the cleaned training set

In [24]:

train_clean = train[~train['review'].map(is_bad_data)]

print(f"Original size: {len(train)}")
print(f"Cleaned size: {len(train_clean)}")



Original size: 6666
Cleaned size: 3999
--- Result after Data Cleaning ---
Accuracy: 97.0%


## Evaluating a model trained on the clean training set

In [25]:
from sklearn import clone

In [26]:
sgd_clf_clean = clone(sgd_clf)

In [27]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [28]:
print("--- Result after Data Cleaning ---")
evaluate(sgd_clf_clean)

--- Result after Data Cleaning ---
Accuracy: 97.1%
